# **Initialization**

In [41]:
"""Start"""

'Start'

In [42]:
#%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import math
import random
import sys
import pulp
import vrplib
import re
import sys
import os
import gc
import contextlib
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from docplex.mp.model import Model

# --- 1. DEFINE PATH TO LIBRARY PARENT FOLDER ---
# Replace this with the ACTUAL path to the folder containing 'didp_ea_lib'
# IMPORTANT: Use r"..." string to handle Windows backslashes correctly
LIBRARY_PARENT_PATH = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\3_Evolutionary_algorithm"
# --- 2. ADD TO SYSTEM PATH ---
if LIBRARY_PARENT_PATH not in sys.path:
    sys.path.append(LIBRARY_PARENT_PATH)
print(f"Library path added: {LIBRARY_PARENT_PATH}")
# --- 3. TEST IMPORT ---
try:
    import evolutionary_algorithm_lib
    from evolutionary_algorithm_lib import *
    from evolutionary_algorithm_lib import (compile_chromosome_to_useable_function, 
                                            combining_modified_didppy_solver_with_chromosome)
    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    
    print("✅ Success! 'evolutionary_algorithm_lib' is imported and ready.")
except ImportError as e:
    print(f"❌ Error: Could not import library. Check the path above.\nDetails: {e}")

Library path added: C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\THESIS_MODIFIED_DIDP\3_Evolutionary_algorithm
✅ Success! 'evolutionary_algorithm_lib' is imported and ready.


# **Configuration & Data input**

In [ ]:
# --- CONFIGURATION ---
# Path to your n20 folder containing .txt files
DATA_DIR = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20"
INPUT_CSV = "TSP_single_dual_bound_results.csv"
OUTPUT_CSV = "batch_ea_results.csv"
LOGS_DIR = "batch_logs"

if not os.path.exists(LOGS_DIR):
    os.makedirs(LOGS_DIR)

# --- GLOBAL VARIABLES (Updated dynamically) ---
current_number_of_customers = 0
current_distance_list = []

# --- BATCH UTILITIES ---
def get_processed_instances(csv_path, logs_dir):
    """
    Returns a set of instances that exist in BOTH the CSV summary and the logs folder.
    """
    # 1. Get instances from CSV
    csv_instances = set()
    if os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if 'Instance' in df.columns:
                csv_instances = set(df['Instance'].unique())
        except:
            pass # CSV read failed, assume empty

    # 2. Get instances from Log Files
    log_instances = set()
    if os.path.exists(logs_dir):
        for filename in os.listdir(logs_dir):
            if filename.endswith("_log.txt"):
                # Extract "0.txt" from "0.txt_log.txt"
                instance_name = filename.replace("_log.txt", "")
                log_instances.add(instance_name)

    # 3. Return Intersection (Must be in BOTH to be considered "Done")
    return csv_instances.intersection(log_instances)

def append_result_to_csv(result_dict, csv_path):
    df = pd.DataFrame([result_dict])
    df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

print("✅ Configuration set.")

def read_tsp_cappart_format(file_path):
    with open(file_path, 'r') as f:
        values = f.read().split()
    iterator = iter(values)
    n = int(next(iterator))
    c = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(int(float(next(iterator))))
        c.append(row)
    return n, c

# Cell 3.5: Data Cleanup Utility

def clean_batch_data(csv_path, logs_dir):
    """
    Ensures consistency between the CSV summary and the Log files.
    1. Removes duplicate instances in CSV (keeps last).
    2. Removes CSV rows if the corresponding Log file is missing.
    3. Deletes Log files if the corresponding CSV row is missing.
    """
    print("🧹 Starting Data Cleanup...")
    
    # 1. Load CSV
    if not os.path.exists(csv_path):
        print("   -> CSV not found. Nothing to clean in CSV.")
        # If CSV missing but logs exist, we might want to clear logs, 
        # but usually better to leave them or delete manually to be safe.
        return 

    try:
        df = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("   -> CSV is empty.")
        return

    original_count = len(df)
    
    # 2. Deduplicate CSV (Keep the last run)
    df.drop_duplicates(subset=['Instance'], keep='last', inplace=True)
    dedup_count = len(df)
    if original_count > dedup_count:
        print(f"   -> Removed {original_count - dedup_count} duplicate rows from CSV.")

    # 3. Remove CSV rows without matching Log files
    valid_indices = []
    instances_in_csv = set()
    
    for index, row in df.iterrows():
        instance_name = row['Instance']
        expected_log = os.path.join(logs_dir, f"{instance_name}_log.txt")
        
        if os.path.exists(expected_log):
            valid_indices.append(index)
            instances_in_csv.add(instance_name)
        else:
            print(f"   -> Removing CSV row for '{instance_name}' (Log file missing).")
            
    # Filter dataframe to keep only valid rows
    df_clean = df.loc[valid_indices]
    
    # Save cleaned CSV
    df_clean.to_csv(csv_path, index=False)
    print(f"   -> CSV saved. Rows: {len(df_clean)} (was {original_count}).")

    # 4. Remove Orphan Log files (Log exists, but not in CSV)
    if os.path.exists(logs_dir):
        files = os.listdir(logs_dir)
        for filename in files:
            if filename.endswith("_log.txt"):
                instance_from_log = filename.replace("_log.txt", "")
                
                if instance_from_log not in instances_in_csv:
                    file_path = os.path.join(logs_dir, filename)
                    try:
                        os.remove(file_path)
                        print(f"   -> Deleted orphan log: {filename} (Not in CSV).")
                    except OSError as e:
                        print(f"   -> Error deleting {filename}: {e}")

    print("✨ Data Cleanup Complete.\n")


✅ Configuration set.


# **Model and dual bounds declaration**

In [44]:
def tsp_model_registry():
    # Uses globals: current_number_of_customers, current_distance_list
    n = current_number_of_customers
    c = current_distance_list
    
    model = m_dp.Model(maximize=False, float_cost=False)
    customer = model.add_object_type(number=n)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    location = model.add_element_var(object_type=customer, target=0)
    travel_time = model.add_int_table(c)

    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}", cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
            preconditions=[unvisited.contains(j)], effects=[(unvisited, unvisited.remove(j)), (location, j)],
        )
        model.add_transition(visit)

    return_to_depot = m_dp.Transition(
        name="return", cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
        effects=[(location, 0)], preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)
    model.add_base_case([unvisited.is_empty(), location == 0])

    metadata = {"num_nodes": n, "distance_matrix": c, "unvisited_var": unvisited, "location_var": location}
    return (model, metadata)

def tsp_dual_bound_registry(didp_bundle):
    model, metadata = didp_bundle
    dist_matrix = np.array(metadata['distance_matrix'])
    unvisited_var = metadata['unvisited_var']
    
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = dist_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = dist_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        np.fill_diagonal(assign_mat, np.inf)
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    return automatic_creation_of_dual_bounds_registry(locals())

print("✅ TSP Model defined.")

✅ TSP Model defined.


# **Execution**

In [45]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 2        # Size of the population in each generation
GENERATIONS = 5           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 5 #seconds

In [46]:
# --- EXECUTE CLEANUP ---
# Run this right before your main loop
clean_batch_data(OUTPUT_CSV, LOGS_DIR)
df_input = pd.read_csv(INPUT_CSV)
processed = get_processed_instances(OUTPUT_CSV, LOGS_DIR)

if len(processed) < len(df_input):
    print(f"🚀 Ready to start batch Run. {len(processed)}/{len(df_input)} instances already completed.")
    print(f"🚀 Starting batch Run. {len(processed)}/{len(df_input)} instances already completed.")
else:
    print(f"🚀 {len(processed)}/{len(df_input)} instances already completed. No need further running")
    
# Modified check in Cell 4
for index, row in df_input.iterrows():
    instance_name = row['Instance']
    log_file_path = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
    
    # Check BOTH the CSV record AND the actual log file
    if instance_name in processed and os.path.exists(log_file_path):
        continue # Safe to skip

    optimal_cost = row['Cost']
    file_path = os.path.join(DATA_DIR, instance_name)
    print(f"\nProcessing {instance_name} (Ref Cost: {optimal_cost})...")

    try:
        # A. Update Global Data for this instance
        current_number_of_customers, current_distance_list = read_tsp_cappart_format(file_path)
        
        # B. Configure Params
        params = EAHyperparameters(
            # --- 1. Population ---
            population_size=POPULATION_SIZE,          
            generations=GENERATIONS,
            crossover_rate=CROSSOVER_RATE,
            mutation_rate=MUTATION_RATE,
            elitism_rate=ELITISM_RATE,           

            # --- 2. Ranges & Constraints ---
            lb_range_of_constant=LB_range_of_constant,
            ub_range_of_constant=UB_range_of_constant,
            min_chromosome_length=min_chromosome_length,     
            max_chromosome_length=max_chromosome_length,   

            # --- 3. Operator Specifics ---
            tournament_size=random.randint(2, 10),                             
            tournament_probability=tournament_probability,                    
            mutation_max_subtree_depth=random.randint(min_chromosome_length, max_chromosome_length),                
            homology_1_point_crossover_probability=homology_1_point_crossover_probability,    
            subtree_crossover_probability=subtree_crossover_probability,             
            uniform_crossover_probability=uniform_crossover_probability,             

            # --- 4. Problem Specific ---
            reference_point=optimal_cost,         
            solver_time_limit=SOLVER_TIME_LIMIT,
            
            # Optional: You can override available operations if needed
            available_operations=["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]
        )

        # C. Run EA (Redirecting output to file to keep notebook clean)
        log_file = os.path.join(LOGS_DIR, f"{instance_name}_log.txt")
        start_time = time.time()
        
        # Capture print outputs to log file
        with open(log_file, "w", encoding="utf-8") as f:
            with contextlib.redirect_stdout(f):
                best_ind = evolution_algorithm_execution(
                    didp_model_registry=tsp_model_registry,
                    dual_bound_expression_function=tsp_dual_bound_registry,
                    params=params
                )
                # --- NEW CODE: Print best_ind here to save it to the log ---
                print("\n" + "="*40)
                print("FINAL BEST INDIVIDUAL")
                print("="*40)
                print(best_ind) 
                # -----------------------------------------------------------
        
        total_time = time.time() - start_time

        # D. Save Results
        result_data = {
            "Instance": instance_name,
            "Total_Time_(s)": round(total_time, 2),
            "Best_Fitness": best_ind['fitness'],
            "Best_Chromosome": str(best_ind['chromosome']),
            "Log_File": log_file
        }
        append_result_to_csv(result_data, OUTPUT_CSV)
        print(f"   ✅ Finished! Best Fit: {best_ind['fitness']:.4f} | Time: {total_time:.2f}s")

    except Exception as e:
        print(f"   ❌ Failed: {e}")
    
    finally:
        # E. Cleanup Memory
        gc.collect()

print("\n🎉 Batch Run Complete!")

🧹 Starting Data Cleanup...
   -> CSV not found. Nothing to clean in CSV.
🚀 Ready to start batch Run. 0/20 instances already completed.
🚀 Starting batch Run. 0/20 instances already completed.

Processing 98.txt (Ref Cost: 356)...
   ✅ Finished! Best Fit: 0.0983 | Time: 75.72s

Processing 91.txt (Ref Cost: 365)...
   ✅ Finished! Best Fit: 0.0575 | Time: 50.23s

Processing 18.txt (Ref Cost: 384)...
   ✅ Finished! Best Fit: 0.3307 | Time: 56.22s

Processing 12.txt (Ref Cost: 399)...
   ✅ Finished! Best Fit: 0.2531 | Time: 40.18s

Processing 49.txt (Ref Cost: 360)...
   ✅ Finished! Best Fit: 0.0000 | Time: 25.13s

Processing 99.txt (Ref Cost: 322)...
   ✅ Finished! Best Fit: 0.0000 | Time: 70.26s

Processing 85.txt (Ref Cost: 371)...
   ✅ Finished! Best Fit: 0.4906 | Time: 48.75s

Processing 26.txt (Ref Cost: 389)...
   ✅ Finished! Best Fit: 0.0000 | Time: 69.90s

Processing 1.txt (Ref Cost: 390)...
   ✅ Finished! Best Fit: 0.0000 | Time: 52.80s

Processing 53.txt (Ref Cost: 425)...
   ✅ Fi